# Information
## How to
1. Set the parameters. **UPPERCASE** letters are user input variables
2. Run the reprojection cell


# Requirements

In [3]:
import os
import rasterio
import numpy as np
from pathlib import Path
from tqdm import tqdm

from beak.utilities.transformation import scale_raster, _scale_raster_core
from beak.utilities.io import data_folder, create_file_list, check_path, save_raster


# Scaling

**Scaling** all numerical folders within a specified model configuration.<br>
Reads the <code>ROOT_FOLDER</code> and takes the <code>NUMERICAL</code> subfolder within each model configuration.

**User inputs**

In [4]:
BASE_PATH = data_folder()
EPSG_CODE = 102008
RESOLUTION = 500

BASE_EXTENT = "tusk_great_basin"
BASE_RASTER = BASE_PATH / "processed" / str(f"regional_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "base_raster" / "template_raster.tif"
BASE_EVIDENCE = "geology"
CMA_EXTENT = "regional"

METHOD = "standard"

# Export
# If some data sets are already **LOG** scaled they need special actions. They will be unified and stored in a separate folder.
PATH_INPUT = BASE_PATH / "processed" / str(f"{CMA_EXTENT}_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "unified_imputed" / BASE_EVIDENCE
PATH_EXPORT = BASE_PATH / "processed" / str(f"{CMA_EXTENT}_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "unified_imputed_scaled_std" / BASE_EVIDENCE

OUT_FOLDER = PATH_EXPORT

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Input folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_102008_500\unified_imputed\geology
Export folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_102008_500\unified_imputed_scaled_std\geology


Select subfolders to be scaled.

In [5]:
# Get the list of folders
root_folder = PATH_INPUT
folders = os.listdir(root_folder)

for folder in folders:
  if os.path.isdir(os.path.join(root_folder, folder)):
    print(folder)
  else:
    print(f"No subfolders existing but found {len(folders)} files.")
    break


age
bge
distance


In [6]:
# Select the folders to process
SELECTION = ["age",
             "bge",
             "distance",
             ]

input_folders = [root_folder / folder for folder in SELECTION]

print("Selected folders:")
for folder in input_folders:
  print(folder)

Selected folders:
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_102008_500\unified_imputed\geology\age
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_102008_500\unified_imputed\geology\bge
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_102008_500\unified_imputed\geology\distance


**Run Scaling** for STD scaler
Existing files will **not** be overwritten

In [8]:
# Create file list
file_list = []
for folder in input_folders:
  files = create_file_list(folder, recursive=True)
  file_list.extend(files)

# Show results
print(f"Found {len(file_list)} files.")

Found 5 files.


In [9]:
for file in tqdm(file_list, total=len(file_list)):
    folder_relative = os.path.relpath(file.parent, PATH_INPUT)

    raster = rasterio.open(file)
    raster_array = _scale_raster_core(raster=raster, method=METHOD)

    output_folder = OUT_FOLDER / str(folder_relative)
    out_path = output_folder / file.name
    check_path(Path(os.path.dirname(out_path)))
    save_raster(out_path, array=raster_array, dtype="float32", metadata=raster.meta)


100%|██████████| 5/5 [00:00<00:00,  6.36it/s]
